# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = None` (giữ toàn bộ bài hợp lệ trong 5.000 dòng Golden scope)
- `LAB_MAX_CHUNKS = 10000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [ ]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx spacy datasets langchain-community llama-index

In [1]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path.cwd() / ".env")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", get_secret("NEO4J_USERNAME", "neo4j"))
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

IS_COLAB = Path("/content").exists()
PROJECT_ROOT = Path("/content") if IS_COLAB else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = str(DATA_DIR / "hackernoon_subset.csv")
LAB_MAX_ARTICLES = None
LAB_MAX_CHUNKS = 10000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

/home/fiores/Documents/hien-selflearning/AI_in_Action_Labs/K4-Track3-Lab19-GraphRAG-2A202601812-TranMinhHien/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- Stream đúng 5.000 raw rows để khớp phạm vi của Golden Dataset chính thức (`pandas index 0-4999`).
- `LIMIT_MB = 300` chỉ là hard-stop an toàn thứ hai nếu record bất thường lớn.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Trên Colab file nằm dưới `/content/data/`; khi chạy local file nằm trong `data/` của repo.

In [2]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = DATA_PATH

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
# Raw rows include company-only records; article cap is applied after standardization.
LIMIT_ROWS = 5_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = False

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...


Đang ghi dữ liệu vào: /home/fiores/Documents/hien-selflearning/AI_in_Action_Labs/K4-Track3-Lab19-GraphRAG-2A202601812-TranMinhHien/data/hackernoon_subset.csv


Đang tải (row):   0%|          | 0/5000 [00:00<?, ?row/s]

Đang tải (row): 100%|██████████| 5000/5000 [00:00<00:00, 117619.29row/s]


[DỪNG] Đã đạt giới hạn số dòng: 5,000 dòng (Dung lượng: 2.92 MB)
✅ Hoàn thành: /home/fiores/Documents/hien-selflearning/AI_in_Action_Labs/K4-Track3-Lab19-GraphRAG-2A202601812-TranMinhHien/data/hackernoon_subset.csv
   Rows: 5,000
   Size: 2.92 MB


In [2]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.


✅ Schema ready.


In [3]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid", "url"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").reset_index(drop=True)
    # Compact, deterministic IDs; including dedup_key prevents repeated URLs from
    # producing duplicate chunk IDs and overwriting provenance downstream.
    df["article_id"] = [
        sha1(f"{article_id}\n{dedup_key}")[:20]
        for article_id, dedup_key in zip(df["article_id"], df["dedup_key"])
    ]
    df = df.drop(columns="dedup_key")
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
news_df.to_parquet(DATA_DIR / "news_preprocessed.parquet", index=False)
chunks_df.to_parquet(DATA_DIR / "chunks.parquet", index=False)
print({"raw_rows": len(raw_df), "articles_after_dedup": len(news_df), "chunks": len(chunks_df)})
display(chunks_df.head())

Exact dedup: 2,675 -> 2,105


Chunking:   0%|          | 0/2105 [00:00<?, ?it/s]

Chunking: 100%|██████████| 2105/2105 [00:00<00:00, 94814.27it/s]

{'raw_rows': 5000, 'articles_after_dedup': 2105, 'chunks': 2105}


,chunk_id,article_id,title,published_date,text
0,f8148ed9ce69f8da57e3::c0000,f8148ed9ce69f8da57e3,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,4451fbbb48f69616a0cf::c0000,4451fbbb48f69616a0cf,Adobe student receives national Information and Technology award,2023-05-02,ELKO — An eighth grader at Adobe Middle School is one of 34 middle school aged girls in Nevada to be recognized by t...
2,fdba75c639ca72ce8890::c0000,fdba75c639ca72ce8890,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
3,a1d72a50f7947913234e::c0000,a1d72a50f7947913234e,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
4,07772802e4d7ece8fa2e::c0000,07772802e4d7ece8fa2e,Synex Renewable Energy Corporation (Formerly Synex International Inc.) Third Quarter of Fiscal 2023,2023-05-15,The conference will bring together growth oriented publicly traded clean energy and technology companies ... up to 4...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [4]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a = text.find("{")
    if a < 0:
        raise ValueError("No JSON object found.")
    obj, _ = json.JSONDecoder().raw_decode(text[a:])
    return obj

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
                "max_completion_tokens": 700,
            }
            if str(model).startswith("qwen/"):
                kwargs["reasoning_effort"] = "none"
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            # Some Groq models can emit JSON but do not support server-side JSON mode.
            # Let groq_json() retry once without response_format instead of repeating
            # the same deterministic 400 validation error.
            if getattr(e, "status_code", None) == 429:
                msg = str(e)
                ms = re.search(r"try again in ([0-9.]+)ms", msg, flags=re.I)
                sec = re.search(r"try again in ([0-9.]+)s", msg, flags=re.I)
                wait_s = float(ms.group(1)) / 1000 if ms else (float(sec.group(1)) if sec else None)
                if wait_s is not None and wait_s <= 60 and attempt < max_retries - 1:
                    time.sleep(wait_s + 1.0)
                    continue
                raise
            if json_mode and (getattr(e, "status_code", None) == 400 or "json_validate_failed" in str(e)):
                raise
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user}]
    try:
        text, usage = groq_chat(messages, model=model, json_mode=True)
    except Exception as e:
        if getattr(e, "status_code", None) != 400 and "json_validate_failed" not in str(e):
            raise
        text, usage = groq_chat(messages, model=model, json_mode=False)
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [5]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5, request_interval_s=10):
    if chunks_subset.empty:
        return pd.DataFrame(columns=["chunk_id", "resolved_text", "unresolved_mentions"])
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        error_marker = None
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception as e:
            error_marker = f"COREF_BATCH_FAILED:{type(e).__name__}:{str(e)[:300]}"
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [[error_marker] for _ in range(len(batch))],
            })
        out.append(df)
        if error_marker and ('RateLimitError' in error_marker or '429' in error_marker):
            remaining = chunks_subset.iloc[start + len(batch):]
            if not remaining.empty:
                out.append(pd.DataFrame({
                    "chunk_id": remaining["chunk_id"].tolist(),
                    "resolved_text": remaining["text"].tolist(),
                    "unresolved_mentions": [[error_marker] for _ in range(len(remaining))],
                }))
            break
        if request_interval_s and start + len(batch) < len(chunks_subset):
            time.sleep(request_interval_s)
    return pd.concat(out, ignore_index=True)

extraction_source = (
    chunks_df.sample(
        n=min(EXTRACTION_MAX_CHUNKS, len(chunks_df)), random_state=SEED
    ).sort_index().reset_index(drop=True)
)
coref_pattern = re.compile(
    r"\b(?:it|they|them|their|he|him|his|she|her|hers|the company|the startup)\b",
    flags=re.I,
)
needs_coref = extraction_source.text.fillna("").str.contains(coref_pattern)
coref_candidates_all = extraction_source[needs_coref].copy()
coref_passthrough = pd.DataFrame({
    "chunk_id": extraction_source.loc[~needs_coref, "chunk_id"],
    "resolved_text": extraction_source.loc[~needs_coref, "text"],
    "unresolved_mentions": [[] for _ in range((~needs_coref).sum())],
})
coref_checkpoint = DATA_DIR / "extraction_source_resolved.parquet"
coref_previous = pd.DataFrame(columns=["chunk_id", "resolved_text", "unresolved_mentions"])
if coref_checkpoint.exists():
    previous = pd.read_parquet(coref_checkpoint)
    if {"chunk_id", "resolved_text", "unresolved_mentions"}.issubset(previous.columns):
        ok = ~previous.unresolved_mentions.map(
            lambda xs: any(str(x).startswith("COREF_BATCH_FAILED") for x in xs)
        )
        valid_ids = set(coref_candidates_all.chunk_id)
        coref_previous = previous.loc[ok & previous.chunk_id.isin(valid_ids), [
            "chunk_id", "resolved_text", "unresolved_mentions"
        ]].drop_duplicates("chunk_id")
coref_candidates = coref_candidates_all[
    ~coref_candidates_all.chunk_id.isin(coref_previous.chunk_id)
].copy()
coref_new = run_coref(coref_candidates, batch_size=5, request_interval_s=10)
coref_resolved = pd.concat([coref_previous, coref_new], ignore_index=True)
coref_df = pd.concat([coref_resolved, coref_passthrough], ignore_index=True)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")
extraction_source.to_parquet(DATA_DIR / "extraction_source_resolved.parquet", index=False)
coref_failed = coref_df.unresolved_mentions.map(
    lambda xs: any(str(x).startswith("COREF_BATCH_FAILED") for x in xs)
).sum()
print({
    "extraction_chunks": len(extraction_source),
    "llm_coref_candidates": len(coref_candidates_all),
    "resumed_coref_rows": len(coref_previous),
    "conservative_passthrough": len(coref_passthrough),
    "coref_failed_chunks": int(coref_failed),
})
if coref_failed:
    raise RuntimeError(f"Coreference failed for {coref_failed} chunks; stop before graph extraction.")
display(coref_df.head())

{'extraction_chunks': 400, 'llm_coref_candidates': 137, 'resumed_coref_rows': 137, 'conservative_passthrough': 263, 'coref_failed_chunks': 0}


,chunk_id,resolved_text,unresolved_mentions
0,7a4a6cd67a095001d8db::c0000,It''s surprising how many different times a life has been saved by technology ... Technology is capable of advanced ...,[It (first occurrence)]
1,80b83b7c226784f4f1fb::c0000,Iridium Communications Inc. (Nasdaq: IRDM) announced Iridium Communications Inc. has created a first-of-its-kind pro...,[]
2,405cc9b946478641117d::c0000,The number of video streaming services available has increased dramatically over the past few years as everyone incl...,[]
3,5f1a9cea4a69f0e59f67::c0000,That’s something that the team at Amazon Web Services (AWS) fully understands. Through Amazon Web Services' Artifici...,[]
4,2d9dda4b415d1ffe5196::c0000,Prior to joining Ventiv Nichols was most recently Chief Operating Officer for an Insurtech software firm Origami Ris...,[]


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [6]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=5, request_interval_s=10, batch_starts=None):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    starts = list(batch_starts) if batch_starts is not None else list(range(0, len(source_df), batch_size))
    for pos, start in enumerate(tqdm(starts, desc="NER+RE")):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            if request_interval_s and pos + 1 < len(starts):
                time.sleep(request_interval_s)
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })
        if request_interval_s and pos + 1 < len(starts):
            time.sleep(request_interval_s)

    return pd.DataFrame(triples), pd.DataFrame(errors)

batch_size = 5
all_batch_starts = list(range(0, len(extraction_source), batch_size))
triples_checkpoint = DATA_DIR / "raw_triples.parquet"
errors_checkpoint = OUTPUT_DIR / "extraction_errors.csv"
success_checkpoint = OUTPUT_DIR / "extraction_successful_batches.csv"
previous_triples = pd.read_parquet(triples_checkpoint) if triples_checkpoint.exists() else pd.DataFrame()
if success_checkpoint.exists():
    successful_starts = set(pd.read_csv(success_checkpoint)["start"].astype(int))
elif errors_checkpoint.exists():
    previous_failed = set(pd.read_csv(errors_checkpoint)["start"].astype(int))
    successful_starts = set(all_batch_starts) - previous_failed
else:
    successful_starts = set()
pending_starts = [x for x in all_batch_starts if x not in successful_starts]
new_triples_df, extraction_errors_df = run_extraction(
    extraction_source, batch_size=batch_size, request_interval_s=10, batch_starts=pending_starts
)
failed_starts = set(extraction_errors_df["start"].astype(int)) if len(extraction_errors_df) else set()
successful_starts.update(set(pending_starts) - failed_starts)
pd.DataFrame({"start": sorted(successful_starts)}).to_csv(success_checkpoint, index=False)
raw_triples_df = pd.concat([previous_triples, new_triples_df], ignore_index=True)
if len(raw_triples_df):
    raw_triples_df = raw_triples_df.drop_duplicates().reset_index(drop=True)
raw_triples_df.to_parquet(DATA_DIR / "raw_triples.parquet", index=False)
extraction_errors_df.to_csv(OUTPUT_DIR / "extraction_errors.csv", index=False)
print({
    "resumed_successful_batches": len(set(all_batch_starts) - set(pending_starts)),
    "pending_batches_this_run": len(pending_starts),
    "total_successful_batches": len(successful_starts),
    "triples": len(raw_triples_df),
    "failed_batches": len(extraction_errors_df),
})
if len(extraction_errors_df):
    raise RuntimeError(f"NER/RE failed for {len(extraction_errors_df)} batches; stop before Neo4j ingestion.")
display(raw_triples_df.head())

NER+RE: 0it [00:00, ?it/s]

NER+RE: 0it [00:00, ?it/s]

{'resumed_successful_batches': 80, 'pending_batches_this_run': 0, 'total_successful_batches': 80, 'triples': 117, 'failed_batches': 0}


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Renovus,Company,INVESTED_IN,Aretum,Company,c675bf02bc305c7c61cb::c0000,2023-04-19,Aretum becomes the fourth company backed by Renovus,1.0
1,Barry Holt,Person,WORKED_AT,Information Services Group,Company,e0f0f16d161f09ff6a6a::c0000,2023-08-05,Barry Holt; Senior Advisor; Information Services Group,1.0
2,Synchronoss,Company,DEVELOPED,Personal Cloud platform,Technology,5e6686cccdaea1fcc480::c0000,2023-02-22,The new Synchronoss Personal Cloud platform enables telecom operators...,1.0
3,LeanTaaS,Company,ACQUIRED,Hospital IQ,Company,9d83fd83cedaa7d26a2f::c0000,2023-01-10,LeanTaaS Acquires Hospital IQ,1.0
4,Amazon Web Services,Company,DEVELOPED,Artificial Intelligence (AI)/machine learning (ML) technology,Technology,5f1a9cea4a69f0e59f67::c0000,2022-12-29,Amazon Web Services' Artificial Intelligence (AI)/machine learning (ML) technology,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [6]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    ta, tb = set(na.split()), set(nb.split())
    # Product qualifiers are semantically meaningful: Google Store is not
    # Google Play Store; Apple is not Apple Watch.
    if ta < tb or tb < ta:
        return False
    return SequenceMatcher(None, na, nb).ratio() >= 0.85

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j:
                    continue
                ok = merge_guard(names[i], names[j])
                if float(score) < threshold:
                    if float(score) >= 0.60:
                        audit.append({
                            "type": typ, "left": names[i], "right": names[j],
                            "similarity": float(score),
                            "decision": "REJECT_GUARD" if float(score) >= 0.85 and not ok else "REJECT_THRESHOLD"
                        })
                    continue
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
entity_resolution_audit_df.to_csv(OUTPUT_DIR / "entity_resolution_audit.csv", index=False)
triples_df.to_parquet(DATA_DIR / "triples_canonical.parquet", index=False)
print({"canonical_triples": len(triples_df), "audit_rows": len(entity_resolution_audit_df)})
display(entity_resolution_audit_df.head(20))

{'canonical_triples': 117, 'audit_rows': 24}


,type,left,right,similarity,decision
0,Company,Information Services Group,Universal Information Services,0.680568,REJECT_THRESHOLD
1,Company,Amazon Web Services,Amazon Web Services Inc.,0.823628,REJECT_THRESHOLD
2,Company,Integral,Integreon,0.644924,REJECT_THRESHOLD
3,Company,ServiceNow,ServiceNow Inc.,0.811463,REJECT_THRESHOLD
4,Company,Biz Technology Solutions,Biz Technology Solutions Inc.,0.865529,REJECT_THRESHOLD
5,Company,CoolSys,Ansys,0.682937,REJECT_THRESHOLD
6,Company,Syntech,Synopsys Inc.,0.674321,REJECT_THRESHOLD
7,Company,Syntech,TD SYNNEX,0.620259,REJECT_THRESHOLD
8,Company,Hon Hai Technology Group (Foxconn),Future Technology Group,0.636291,REJECT_THRESHOLD
9,Company,ServiceNow Inc.,Amazon Web Services Inc.,0.670567,REJECT_THRESHOLD


In [7]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)
print({"nodes_prepared": len(nodes_df), "edges_prepared": len(triples_df)})

{'nodes_prepared': 195, 'edges_prepared': 117}


In [8]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 195, 'edges': 117, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,0e132222d1315530d6efca52,Google,Company,4
1,9960a1ca5dd13b4f18ca1fb7,New Charter Technologies,Company,3
2,483064d6337d5cd16bda1f38,PSG,Company,3
3,f4b2c186e83e2e224a2527d7,GM,Company,3
4,3a7f599b25d4a0d9a9357b0b,GPT-3,Technology,3
5,c04f99d108d0d5c105c40fa6,David S. Cloud,Person,3
6,8e4ccf66d61b5766bcb7d7bb,Claroty,Company,3
7,6b8d3ec5d642d544e137e070,Future Technology Group,Company,3
8,fb0f4df56fab164ec48722f0,Microsoft,Company,3
9,84f141d5a03e6e7c620a4ff6,Blue Solutions,Company,3


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [8]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   6%|▌         | 1/17 [00:05<01:34,  5.93s/it]

Batches:  12%|█▏        | 2/17 [00:07<00:47,  3.19s/it]

Batches:  18%|█▊        | 3/17 [00:08<00:30,  2.20s/it]

Batches:  24%|██▎       | 4/17 [00:09<00:22,  1.70s/it]

...

Batches:  88%|████████▊ | 15/17 [00:19<00:01,  1.27it/s]

Batches:  94%|█████████▍| 16/17 [00:20<00:00,  1.36it/s]

Batches: 100%|██████████| 17/17 [00:20<00:00,  1.69it/s]

Batches: 100%|██████████| 17/17 [00:20<00:00,  1.20s/it]

Flat vectors: 2105


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [9]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)
print({"entity_matcher_nodes": len(entity_match_store)})

{'entity_matcher_nodes': 195}


In [10]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [11]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [12]:
#@title 4.1 — Golden Dataset chính thức từ upstream
GOLDEN_PATH = str(DATA_DIR / "graphrag_golden_50_first5000_detailed.csv")

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

if not Path(GOLDEN_PATH).exists():
    raise FileNotFoundError(f"Thiếu Golden Dataset chính thức: {GOLDEN_PATH}")
golden_df = pd.read_csv(GOLDEN_PATH)
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

validate_golden(golden_df, require_answers=True)

,id,group,difficulty,question,reference_answer,reference_evidence,evidence_row_ids_0based,evidence_urls_json,expected_hops,seed_entities,required_relations,adversarial_dimension,gold_reasoning,scoring_notes,source_scope
0,G5000-26,multi-hop,hard,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,row 2532 (2023-07-26 20:19:00): Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft...,"[2532, 2537]","[""https://www.reuters.com/technology/amazon-has-drawn-thousands-try-its-ai-service-competing-with-microsoft-google-2...",2,"[""Amazon"", ""Cohere""]","[""PROVIDES_ACCESS_TO"", ""DEVELOPED""]",Duplicate coverage + detail union,Merge the duplicate Amazon reports and retain non-conflicting details from each.,Need Cohere plus the customer-service-agent capability; clinical notes is an additional supported detail.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
1,G5000-27,cross-doc,hard,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,row 3357 (2023-06-01 13:16:00): 3 Best Cloud Stocks to Buy in June | row 2905 (2023-06-14 08:47:00): Exclusive: Amaz...,"[3357, 2905]","[""https://www.fool.com/investing/2023/06/01/3-best-cloud-stocks-to-buy-in-june/"", ""https://www.reuters.com/technolog...",2,"[""AMD"", ""AWS""]","[""POWERS"", ""CONSIDERING""]",General-to-specific relation trap,Distinguish a general market statement from a specific vendor adoption claim.,Critical: do not infer AWS adopted the new AMD AI chips.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
2,G5000-28,multi-hop,hard,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,row 3395 (2023-08-29 18:07:00): Google Cloud Kicks Off Next '23 with a New Way to Cloud,[3395],"[""https://www.tmcnet.com/usubmit/2023/08/29/9871484.htm""]",3,"[""Google Cloud"", ""Meta"", ""Technology Innovation Institute"", ""Anthropic""]","[""HOSTS_MODEL_FROM"", ""PREANNOUNCED""]",Multi-entity model/provider mapping,Build separate provider->model mappings rather than a flat list.,All three providers and their models are required.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
3,G5000-29,cross-doc,hard,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...",row 3380 (2023-07-21 13:01:00): The White House and big tech companies release commitments on managing AI | row 3330...,"[3380, 3330]","[""https://www.wfae.org/united-states-world/united-states-world/2023-07-21/the-white-house-and-big-tech-companies-rel...",2,"[""White House"", ""Google"", ""Meta"", ""OpenAI"", ""IBM"", ""Adobe"", ""Salesforce""]","[""COMMITTED_TO""]",Temporal participant-set expansion,Compare the named participant sets and shared commitment concept over time.,Do not claim the September companies were part of the July seven unless explicitly named.,ONLY first 5000 data rows of hackernoon_subset.csv (pandas index 0-4999)
4,G5000-30,multi-hop,hard,"Meta appears in two different AI contexts in the selected data. What are they, and what distinct relation should the...","At Google Cloud Next, Meta is the provider/source of Llama 2 and Code Llama models made available on Google Cloud. S...",row 3395 (2023-08-29 18:07:00): Google Cloud Kicks Off Next '23 with a New Way to Cloud | row 3380 (2023-

✅ Golden Dataset valid.


In [13]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [14]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = str(OUTPUT_DIR / "graphrag_eval_checkpoint.csv")

def run_evaluation(golden_df):
    rows = []
    if Path(CHECKPOINT).exists():
        previous = pd.read_csv(CHECKPOINT)
        valid_ids = set(golden_df.id.astype(str))
        rows = previous[previous.id.astype(str).isin(valid_ids)].to_dict("records")
    completed_ids = {str(r["id"]) for r in rows}
    pending = golden_df[~golden_df.id.astype(str).isin(completed_ids)]
    print({"resumed_rows": len(rows), "pending_rows": len(pending)})
    for q in tqdm(pending.itertuples(index=False), total=len(pending), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

✅ Golden Dataset valid.
{'resumed_rows': 10, 'pending_rows': 15}


Evaluation:   0%|          | 0/15 [00:00<?, ?it/s]

Evaluation:   7%|▋         | 1/15 [00:02<00:34,  2.44s/it]

Evaluation:  13%|█▎        | 2/15 [00:05<00:36,  2.82s/it]

Evaluation:  20%|██        | 3/15 [00:23<01:58,  9.87s/it]

...

Evaluation:  87%|████████▋ | 13/15 [04:42<00:54, 27.19s/it]

Evaluation:  93%|█████████▎| 14/15 [05:13<00:28, 28.37s/it]

Evaluation: 100%|██████████| 15/15 [05:41<00:00, 28.31s/it]

Evaluation: 100%|██████████| 15/15 [05:41<00:00, 22.76s/it]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - **Question:** What external technology p...,<think>\nHere's a thinking process:\n\n1. **Analyze User Question:**\n - **Question:** What external technology p...,5,5,5,5,5,5,3.598765,4.412753,2202,2579,The candidate answer accurately extracts both the external technology provider (Cohere) and the accompanying AI capa...,The candidate correctly identifies that the supplied context does not contain the requested information regarding a ...,0
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,The statement that AMD powers multiple cloud services [chunk_id=782819c14d10efa74606::c0000] is reconciled with the ...,The graph reconciles these statements by distinguishing between general infrastructure and specific AI hardware cons...,5,5,5,5,5,5,0.642596,0.602387,755,743,The candidate answer accurately synthesizes the two provided context chunks to address the reconciliation question. ...,The candidate answer accurately synthesizes the provided context to reconcile the apparent contradiction. It correct...,0
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,"Based on the provided context, the following model providers and their associated models are connected to Google Clo...","Based on the provided context, the following model providers and their associated models are connected to Google Clo...",5,5,5,5,5,5,0.551758,0.589272,843,809,"The candidate answer correctly identifies all three model providers (Meta, Technology Innovation Institute, Anthropi...","The candidate answer correctly identifies all three model providers (Meta, Technology Innovation Institute, Anthropi...",0
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...","Participation broadened from OpenAI, which made pledges in July, to include IBM, Adobe, Salesforce, and five more te...","Participation broadened from initial pledges made by OpenAI in July to include IBM, Adobe, Salesforce, and five othe...",2,4,5,5,4,5,1.486499,4.514792,714,646,"The candidate answer is faithful to the provided context, correctly identifying the new participants in September an...","The candidate answer accurately reflects the provided context, specifically chunk b65d094feaeb71dd5586, which detail...",0
4,G5000-30,multi-hop,"Meta appears in two different AI contexts in the selected data. What are they, and what distinct relation should the...","At Google Cloud Next, Meta is the provider/source of Llama 2 and Code Llama models made available on Google Cloud. S...","Based on the provided context, there is insufficient evidence to answer the question. The text mentions ""Meta"" only ...","Based on the provided context, Meta appears in only one AI-related context:\n\n1. **Context:** Meta is a company th...",1,1,5,5,1,1,6.6860

In [15]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv(OUTPUT_DIR / "graphrag_eval_results.csv", index=False)
comparison_df.to_csv(OUTPUT_DIR / "graphrag_vs_flatrag_summary.csv", index=False)
print({"evaluation_rows": len(eval_results_df), "summary_rows": len(comparison_df)})

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,2.545,2.455,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,4.636,4.636,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,2.727,2.545,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),3.140,2.584,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,736.364,685.909,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),5.627,4.521,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,753.500,659.500,GraphRAG không đắt hơn trong sample này.


{'evaluation_rows': 25, 'summary_rows': 15}


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [16]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

# test_supernode_policy()
# show_resolution_audit(entity_resolution_audit_df)

## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [17]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [18]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

In [19]:
ALLOWED_NODE_TYPES={'Company','Person','Technology'}
ALLOWED_RELATIONS={'ACQUIRED','DEVELOPED','INVESTED_IN','FOUNDED','WORKED_AT','PARTNERED_WITH','USES','LEADS'}
raw_triples_df=pd.read_parquet(DATA_DIR/'raw_triples.parquet')